# pandas — task-indexed cheatsheet


Look up the idiom you need by **task**, not by function name. Each entry is:

1. The question you're trying to answer.
2. The canonical pattern.
3. *Why this is the right tool* + the most common mistake.

Run any cell to see the pattern in action — every example is self-contained.


---
Setup

Every entry below assumes these imports and a small toy frame are loaded. Run this cell once.


### Setup — run me first


In [1]:
import pandas as pd
import numpy as np

# Synthetic 3-symbol hourly frame with a deliberate gap in 'A' (no 02:00) and a NaN.
ts = pd.to_datetime([
    '2024-01-01 00:00', '2024-01-01 01:00', '2024-01-01 03:00',
    '2024-01-01 00:00', '2024-01-01 01:00', '2024-01-01 02:00',
    '2024-01-01 00:00', '2024-01-01 01:00', '2024-01-01 02:00',
], utc=True)
df = pd.DataFrame({
    'ts':     ts,
    'symbol': ['A', 'A', 'A', 'B', 'B', 'B', 'C', 'C', 'C'],
    'close':  [100.0, 101.0, np.nan, 50.0, 51.0, 52.0, 30.0, 30.5, 32.0],
    'volume': [10, 11, 12, 5, 6, 7, 8, 9, 10],
})
df.head()

,ts,symbol,close,volume
0,2024-01-01 00:00:00+00:00,A,100.0,10
1,2024-01-01 01:00:00+00:00,A,101.0,11
2,2024-01-01 03:00:00+00:00,A,NaN,12
3,2024-01-01 00:00:00+00:00,B,50.0,5
4,2024-01-01 01:00:00+00:00,B,51.0,6


---
## 1. Loading and writing data

How to read into and out of common formats. Parquet is the default for anything bigger than 'small'.


### How do I read a CSV?

Use `pd.read_csv`. For datetime columns, parse them at read time so you don't repeat the conversion later.


In [2]:
# pd.read_csv('path.csv', parse_dates=['ts'])
# from io import StringIO
data = "ts,close\n2024-01-01,100\n2024-01-02,101\n"
from io import StringIO
df_csv = pd.read_csv(StringIO(data), parse_dates=['ts'])
print(df_csv.dtypes)

ts       datetime64[ns]
close             int64
dtype: object


*Why parse_dates at read*: skips a separate `pd.to_datetime` step and lets pandas infer dtype efficiently. *Common mistake*: forgetting `parse_dates` and then trying to use `.dt` accessors — they fail on object dtype.


### How do I read a Parquet file?

Use `pd.read_parquet` with the `pyarrow` engine. Faster, smaller, and column-typed — almost always preferable to CSV for non-trivial data.


In [3]:
# Round-trip a frame to parquet and back.
df.to_parquet('/tmp/demo.parquet', index=False)
df_back = pd.read_parquet('/tmp/demo.parquet')
print(df_back.shape)

(9, 4)


*Why parquet over CSV*: 5-10× smaller files, 2-5× faster reads, and dtypes are preserved (datetimes stay datetimes, floats stay floats). *Common mistake*: using `index=True` then losing the index name on round-trip — set the index explicitly after loading if you want it.


### How do I read just a few columns from a big file?

Pass `usecols=` (CSV) or `columns=` (parquet). Massively reduces memory and read time when you only need a subset.


In [4]:
df.to_parquet('/tmp/demo.parquet', index=False)
just_close = pd.read_parquet('/tmp/demo.parquet', columns=['ts', 'close'])
print(just_close.columns.tolist())

['ts', 'close']


*Common mistake*: reading the full frame and then projecting — pyarrow can push the column projection down to the read, but only if you use the kwarg.


---
## 2. Selecting rows and columns

The boundary between `.loc`, `.iloc`, boolean masks and `.query` is one of the most common time-sinks. Pick the right one for the task.


### How do I pick rows by *label* (timestamp, named index)?

Use `.loc` with the label or a slice of labels. Endpoints are **inclusive** in `.loc` slices (unlike `.iloc`).


In [5]:
df_idx = df.set_index('ts').sort_index()
# All rows on 2024-01-01 between 00 and 01 inclusive.
print(df_idx.loc['2024-01-01 00:00':'2024-01-01 01:00'])

                          symbol  close  volume
ts                                             
2024-01-01 00:00:00+00:00      A  100.0      10
2024-01-01 00:00:00+00:00      B   50.0       5
2024-01-01 00:00:00+00:00      C   30.0       8
2024-01-01 01:00:00+00:00      A  101.0      11
2024-01-01 01:00:00+00:00      B   51.0       6
2024-01-01 01:00:00+00:00      C   30.5       9


*Why .loc*: works with the actual index values you care about (timestamps, names) rather than positions. *Common mistake*: forgetting that `.loc` slicing includes the end point — `.iloc` does not.


### How do I pick rows by *position* (the 5th row)?

Use `.iloc` with integer positions. Endpoints are **exclusive** (Python-style).


In [6]:
print(df.iloc[0:3])     # first three rows
print(df.iloc[-1])      # last row

                         ts symbol  close  volume
0 2024-01-01 00:00:00+00:00      A  100.0      10
1 2024-01-01 01:00:00+00:00      A  101.0      11
2 2024-01-01 03:00:00+00:00      A    NaN      12
ts        2024-01-01 02:00:00+00:00
symbol                            C
close                          32.0
volume                           10
Name: 8, dtype: object


*Why .iloc*: independent of the index — the 5th row is the 5th row regardless of whether the index is timestamps, strings, or anything else. *Common mistake*: using `.iloc` when you meant labels — works on integer indices, breaks on others.


### How do I filter rows by a condition?

Two ways. **Boolean mask** for general conditions. **`.query`** for cleaner syntax when filtering by column values.


In [7]:
# Boolean mask
print(df[df['close'] > 50])

# .query — cleaner for multiple conditions, but doesn't work with all dtypes
print(df.query('close > 50 and symbol == "B"'))

                         ts symbol  close  volume
0 2024-01-01 00:00:00+00:00      A  100.0      10
1 2024-01-01 01:00:00+00:00      A  101.0      11
4 2024-01-01 01:00:00+00:00      B   51.0       6
5 2024-01-01 02:00:00+00:00      B   52.0       7
                         ts symbol  close  volume
4 2024-01-01 01:00:00+00:00      B   51.0       6
5 2024-01-01 02:00:00+00:00      B   52.0       7


*Why both*: boolean masks compose with anything (`mask & other_mask`); `.query` is shorter and uses backticks for column names with spaces. *Common mistake*: chained masks without parentheses — `df[df['a'] > 1 & df['b'] < 2]` parses wrong because `&` binds tighter than comparisons.


### How do I filter where a column is in a list?

Use `.isin()`. Way faster than chained `==` comparisons.


In [8]:
print(df[df['symbol'].isin(['A', 'C'])])

                         ts symbol  close  volume
0 2024-01-01 00:00:00+00:00      A  100.0      10
1 2024-01-01 01:00:00+00:00      A  101.0      11
2 2024-01-01 03:00:00+00:00      A    NaN      12
6 2024-01-01 00:00:00+00:00      C   30.0       8
7 2024-01-01 01:00:00+00:00      C   30.5       9
8 2024-01-01 02:00:00+00:00      C   32.0      10


*Common mistake*: passing a single string instead of a list — `df['symbol'].isin('A')` returns one row per character of 'A', not what you want.


### How do I select specific columns?

Pass a list to `[]`. A bare string returns a Series; a list returns a DataFrame.


In [9]:
print(type(df['close']).__name__)            # 'Series'
print(type(df[['close']]).__name__)          # 'DataFrame'
print(df[['ts', 'symbol', 'close']].head())  # selected columns, in the given order

Series
DataFrame
                         ts symbol  close
0 2024-01-01 00:00:00+00:00      A  100.0
1 2024-01-01 01:00:00+00:00      A  101.0
2 2024-01-01 03:00:00+00:00      A    NaN
3 2024-01-01 00:00:00+00:00      B   50.0
4 2024-01-01 01:00:00+00:00      B   51.0


*Common mistake*: writing `df['close', 'volume']` (no list brackets) — that's a tuple, raises KeyError.


---
## 3. Sorting

Sorting matters for time-series correctness. Many bugs hide in code that *assumes* the index is sorted.


### How do I sort by a column?

`sort_values('col')`. Pass a list for multi-key sorts.


In [10]:
print(df.sort_values('close', ascending=False).head())
print(df.sort_values(['symbol', 'ts']).head())   # multi-key

                         ts symbol  close  volume
1 2024-01-01 01:00:00+00:00      A  101.0      11
0 2024-01-01 00:00:00+00:00      A  100.0      10
5 2024-01-01 02:00:00+00:00      B   52.0       7
4 2024-01-01 01:00:00+00:00      B   51.0       6
3 2024-01-01 00:00:00+00:00      B   50.0       5
                         ts symbol  close  volume
0 2024-01-01 00:00:00+00:00      A  100.0      10
1 2024-01-01 01:00:00+00:00      A  101.0      11
2 2024-01-01 03:00:00+00:00      A    NaN      12
3 2024-01-01 00:00:00+00:00      B   50.0       5
4 2024-01-01 01:00:00+00:00      B   51.0       6


*Common mistake*: forgetting `ascending=False` and reading the wrong end of the data.


### How do I sort by the index?

`sort_index()`. Critical before any `.shift()` or `.rolling()` operation.


In [11]:
df_idx = df.set_index('ts')
print(df_idx.sort_index().head())

                          symbol  close  volume
ts                                             
2024-01-01 00:00:00+00:00      A  100.0      10
2024-01-01 00:00:00+00:00      B   50.0       5
2024-01-01 00:00:00+00:00      C   30.0       8
2024-01-01 01:00:00+00:00      A  101.0      11
2024-01-01 01:00:00+00:00      B   51.0       6


*Why this matters*: `df.shift(1)` operates on the current row order, not by index value. If the index isn't chronological, `shift(1)` shifts to a non-adjacent time. **Always `.sort_index()` before time-aware operations.**


---
## 4. Missing data

`NaN` propagates silently. Catching missingness early prevents downstream confusion.


### How do I count missing values per column?

`df.isna().sum()` or `(df.isna().mean() * 100)` for percentages.


In [12]:
print(df.isna().sum())                 # absolute count
print((df.isna().mean() * 100).round(2))   # percent missing per column

ts        0
symbol    0
close     1
volume    0
dtype: int64
ts         0.00
symbol     0.00
close     11.11
volume     0.00
dtype: float64


*Common mistake*: confusing `.isna()` (returns a boolean frame) with `.isnull()` (alias of the same thing). Use `.isna()` for consistency.


### How do I drop rows with any missing value?

`dropna()`. Use `subset=` to restrict to specific columns.


In [13]:
print(df.dropna())                       # drops any row with any NaN
print(df.dropna(subset=['close']))       # drops only if 'close' is NaN

                         ts symbol  close  volume
0 2024-01-01 00:00:00+00:00      A  100.0      10
1 2024-01-01 01:00:00+00:00      A  101.0      11
3 2024-01-01 00:00:00+00:00      B   50.0       5
4 2024-01-01 01:00:00+00:00      B   51.0       6
5 2024-01-01 02:00:00+00:00      B   52.0       7
6 2024-01-01 00:00:00+00:00      C   30.0       8
7 2024-01-01 01:00:00+00:00      C   30.5       9
8 2024-01-01 02:00:00+00:00      C   32.0      10
                         ts symbol  close  volume
0 2024-01-01 00:00:00+00:00      A  100.0      10
1 2024-01-01 01:00:00+00:00      A  101.0      11
3 2024-01-01 00:00:00+00:00      B   50.0       5
4 2024-01-01 01:00:00+00:00      B   51.0       6
5 2024-01-01 02:00:00+00:00      B   52.0       7
6 2024-01-01 00:00:00+00:00      C   30.0       8
7 2024-01-01 01:00:00+00:00      C   30.5       9
8 2024-01-01 02:00:00+00:00      C   32.0      10


*Why subset*: dropping any-NaN often discards useful rows where only an irrelevant column was missing. *Common mistake*: chaining `dropna` before checking what fraction would be dropped — surprise data loss.


### How do I fill missing values?

`fillna(value)` for a constant. `ffill()` / `bfill()` for forward / backward propagation.


In [14]:
# Constant fill.
print(df['close'].fillna(0).head())

# Forward-fill: carries the last observed value forward (good for prices, bad for returns).
print(df['close'].ffill().head())

0    100.0
1    101.0
2      0.0
3     50.0
4     51.0
Name: close, dtype: float64
0    100.0
1    101.0
2    101.0
3     50.0
4     51.0
Name: close, dtype: float64


*Why ffill is dangerous on returns*: returns should be NaN where there's no price change to compute. Forward-filling fakes a 0% return that didn't happen. Use `ffill` on prices/cumulative quantities, not deltas.


---
## 5. GroupBy — per-group operations

The single most-used pandas idiom. The decision: do you want to **aggregate** (one row per group), **transform** (same shape as input, group-aware values), or **apply** (anything else)?


### How do I compute one statistic per group?

`df.groupby('col').agg('func')` or `.func()`. Returns one row per group.


In [15]:
# One stat per group
print(df.groupby('symbol')['close'].mean())

# Multiple stats at once
print(df.groupby('symbol')['close'].agg(['mean', 'std', 'count']))

symbol
A    100.500000
B     51.000000
C     30.833333
Name: close, dtype: float64
              mean       std  count
symbol                             
A       100.500000  0.707107      2
B        51.000000  1.000000      3
C        30.833333  1.040833      3


*When to use*: "What's the average price per symbol?" → aggregate. *Common mistake*: calling `.mean()` on a GroupBy that includes non-numeric columns — older pandas would silently drop them, newer pandas raises.


### How do I compute a per-group rolling/cumulative statistic, keeping the original shape?

Use `.transform()`. Returns a Series the same length as the input, where each value is the group-aware statistic.


In [16]:
# Subtract per-symbol mean from each row's close (group-mean centering).
df['close_centered'] = df['close'] - df.groupby('symbol')['close'].transform('mean')
print(df[['symbol', 'close', 'close_centered']])

  symbol  close  close_centered
0      A  100.0       -0.500000
1      A  101.0        0.500000
2      A    NaN             NaN
3      B   50.0       -1.000000
4      B   51.0        0.000000
5      B   52.0        1.000000
6      C   30.0       -0.833333
7      C   30.5       -0.333333
8      C   32.0        1.166667


*When to use*: when you want a derived column that respects group boundaries. *Why not `.apply`*: `.transform` is faster and broadcasts back to the original index automatically.


### How do I compute a custom per-group statistic that doesn't fit `.agg` or `.transform`?

`.apply(func)` is the escape hatch. `func` receives the group's frame; whatever it returns becomes the result.


In [17]:
# Largest per-symbol time gap in hours.
gaps_h = (df.groupby('symbol')['ts']
            .apply(lambda s: s.sort_values().diff().dt.total_seconds().div(3600).max()))
print(gaps_h)

symbol
A    2.0
B    1.0
C    1.0
Name: ts, dtype: float64


*Critical detail*: **sort within the lambda** — groupby preserves first-appearance order, not chronological. *Common mistake*: skipping the sort and getting negative timedeltas when rows happen to be out of order.


### How do I count rows per group?

`.size()` (counts all rows) vs `.count()` (counts non-NaN values per column).


In [18]:
print(df.groupby('symbol').size())             # rows per symbol
print(df.groupby('symbol').count())            # non-NaN count per column per symbol

symbol
A    3
B    3
C    3
dtype: int64
        ts  close  volume  close_centered
symbol                                   
A        3      2       3               2
B        3      3       3               3
C        3      3       3               3


*Common mistake*: using `.count()` and getting confused when the result is per-column rather than a single number per group.


### How do I get the first / last row per group?

`.first()` / `.last()` aggregate; `.head(n)` / `.tail(n)` keep the original frame shape.


In [19]:
print(df.groupby('symbol').first())          # first row per symbol — one row per group
print(df.groupby('symbol').head(2))          # first 2 rows per symbol — original shape

                              ts  close  volume  close_centered
symbol                                                         
A      2024-01-01 00:00:00+00:00  100.0      10       -0.500000
B      2024-01-01 00:00:00+00:00   50.0       5       -1.000000
C      2024-01-01 00:00:00+00:00   30.0       8       -0.833333
                         ts symbol  close  volume  close_centered
0 2024-01-01 00:00:00+00:00      A  100.0      10       -0.500000
1 2024-01-01 01:00:00+00:00      A  101.0      11        0.500000
3 2024-01-01 00:00:00+00:00      B   50.0       5       -1.000000
4 2024-01-01 01:00:00+00:00      B   51.0       6        0.000000
6 2024-01-01 00:00:00+00:00      C   30.0       8       -0.833333
7 2024-01-01 01:00:00+00:00      C   30.5       9       -0.333333


*When to use which*: `.first()` for a snapshot table; `.head(n)` when you want to filter the original frame.


---
## 6. Rolling and expanding windows

The shift-then-rolling order is THE most important time-series correctness habit. Get this wrong and you leak the future into your features.


### How do I compute a rolling mean / std / sum?

`series.rolling(window).mean()` etc. Window is in **rows**, not seconds.


In [20]:
df_idx = df.set_index('ts').sort_index()
btc = df_idx[df_idx['symbol'] == 'B']['close']
print(btc.rolling(2).mean())

ts
2024-01-01 00:00:00+00:00     NaN
2024-01-01 01:00:00+00:00    50.5
2024-01-01 02:00:00+00:00    51.5
Name: close, dtype: float64


*Common mistake*: passing window as a duration ('1H') without first converting the index to a DatetimeIndex — pandas will raise.


### How do I compute a STRICTLY PAST rolling mean (no leakage)?

`series.shift(1).rolling(window).mean()` — shift FIRST, then aggregate.


In [21]:
btc = df.set_index('ts').sort_index().query('symbol == "B"')['close']
past_only = btc.shift(1).rolling(2).mean()    # at row t, uses [t-2, t-1]
print(past_only)

ts
2024-01-01 00:00:00+00:00     NaN
2024-01-01 01:00:00+00:00     NaN
2024-01-01 02:00:00+00:00    50.5
Name: close, dtype: float64


*Why shift first*: `rolling(window).mean()` includes the current row by default. For features that will be used to predict the *current* row's target, the current row is leakage. **Always shift before rolling for predictive features.** *Common mistake*: rolling-then-shift — works mathematically but is fragile (one missing shift and you leak).


### How do I compute a per-group rolling stat?

`df.groupby('symbol')['col'].rolling(w).mean()` — but watch the resulting MultiIndex.


In [22]:
roll = df.groupby('symbol')['close'].rolling(2).mean()
print(roll)                                   # MultiIndex (symbol, original_index)
print(roll.reset_index(0, drop=True).sort_index())   # back to flat index

symbol   
A       0       NaN
        1    100.50
        2       NaN
B       3       NaN
        4     50.50
        5     51.50
C       6       NaN
        7     30.25
        8     31.25
Name: close, dtype: float64
0       NaN
1    100.50
2       NaN
3       NaN
4     50.50
5     51.50
6       NaN
7     30.25
8     31.25
Name: close, dtype: float64


*Common mistake*: forgetting to drop the symbol level — the MultiIndex makes downstream alignment hard.


### How do I compute an expanding (cumulative) statistic?

`series.expanding().mean()` — window grows from the start to the current row.


In [23]:
print(btc.expanding().mean())

ts
2024-01-01 00:00:00+00:00    50.0
2024-01-01 01:00:00+00:00    50.5
2024-01-01 02:00:00+00:00    51.0
Name: close, dtype: float64


*When to use*: cumulative stats where you want all history (e.g. running max, running mean for drift detection).


### How do I require a minimum number of observations in the window?

`min_periods=` — the window returns NaN until at least that many non-NaN values are present.


In [24]:
print(btc.rolling(3, min_periods=1).mean())   # NaN-free, but early values are noisy
print(btc.rolling(3, min_periods=3).mean())   # NaN at start, only full windows

ts
2024-01-01 00:00:00+00:00    50.0
2024-01-01 01:00:00+00:00    50.5
2024-01-01 02:00:00+00:00    51.0
Name: close, dtype: float64
ts
2024-01-01 00:00:00+00:00     NaN
2024-01-01 01:00:00+00:00     NaN
2024-01-01 02:00:00+00:00    51.0
Name: close, dtype: float64


*When to use which*: `min_periods=window` (the default) for honest stats; `min_periods=1` when you'd rather have an early estimate than NaN.


---
## 7. Resample — change the frequency

`resample` is for changing the time-frequency of a series (hourly → daily). It's similar to `groupby` but on a regular time grid.


### How do I aggregate hourly data to daily?

`series.resample('D').agg('func')` or `.func()`.


In [25]:
# Hourly → daily mean for symbol B.
btc = df.set_index('ts').sort_index().query('symbol == "B"')['close']
daily = btc.resample('D').mean()
print(daily)

ts
2024-01-01 00:00:00+00:00    51.0
Freq: D, Name: close, dtype: float64


*Why resample over groupby*: resample fills *missing periods* (a day with no data shows up as NaN, not as missing). Groupby just iterates over present groups.


### How do I get OHLC bars from a tick/minute series?

`series.resample('1H').ohlc()` returns Open/High/Low/Close in one call.


In [26]:
ohlc = btc.resample('1H').ohlc()
print(ohlc)

                           open  high   low  close
ts                                                
2024-01-01 00:00:00+00:00  50.0  50.0  50.0   50.0
2024-01-01 01:00:00+00:00  51.0  51.0  51.0   51.0
2024-01-01 02:00:00+00:00  52.0  52.0  52.0   52.0


/tmp/ipykernel_2705001/4183287950.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  ohlc = btc.resample('1H').ohlc()


*When to use*: any time you have higher-frequency data and need lower-frequency bars. *Common mistake*: forgetting to set the time column as the index first — resample works on the index.


### How do I upsample (lower frequency → higher frequency)?

`resample('h').ffill()` or `.interpolate()`.


In [27]:
daily = btc.resample('D').mean()
hourly = daily.resample('1h').ffill()         # carry forward
print(hourly.head(5))

ts
2024-01-01 00:00:00+00:00    51.0
Freq: h, Name: close, dtype: float64


*Common mistake*: forgetting to fill — pure upsampling produces NaN for the inserted intermediate periods.


### What's the difference between resample and groupby?

`groupby` groups by *value*; `resample` groups by a *regular time interval*. Resample fills missing intervals; groupby skips them.


In [28]:
# groupby on day-of-year (skips days with no data)
print(btc.groupby(btc.index.dayofyear).mean())

# resample('D') (regular grid; missing days become NaN)
print(btc.resample('D').mean())

ts
1    51.0
Name: close, dtype: float64
ts
2024-01-01 00:00:00+00:00    51.0
Freq: D, Name: close, dtype: float64


*Rule of thumb*: regular time grid → resample. Categorical or irregular grouping → groupby.


---
## 8. Reshape — long ↔ wide

The most common feature-engineering operation. **Pivot to wide** for cross-asset features; **melt to long** for tidy plotting.


### How do I go from long format (one row per (symbol, ts)) to wide (one column per symbol)?

`df.pivot(index='ts', columns='symbol', values='close')`. Strict: fails on duplicates.


In [29]:
wide = df.drop_duplicates(['ts', 'symbol']).pivot(
    index='ts', columns='symbol', values='close',
).sort_index()
print(wide)

symbol                         A     B     C
ts                                          
2024-01-01 00:00:00+00:00  100.0  50.0  30.0
2024-01-01 01:00:00+00:00  101.0  51.0  30.5
2024-01-01 02:00:00+00:00    NaN  52.0  32.0
2024-01-01 03:00:00+00:00    NaN   NaN   NaN


*Why .sort_index() afterwards*: pivot is sorted by default in current pandas, but `.sort_index()` is a defensive habit so future code is correct *because of itself*, not because of pandas defaults. *Common mistake*: pivot with duplicate (index, columns) raises a ValueError — use `pivot_table` (with `aggfunc=`) if duplicates are expected.


### How do I go from wide back to long?

`df.melt()`. The inverse of pivot.


In [30]:
long = wide.reset_index().melt(id_vars='ts', var_name='symbol', value_name='close').dropna()
print(long.head(6))

                         ts symbol  close
0 2024-01-01 00:00:00+00:00      A  100.0
1 2024-01-01 01:00:00+00:00      A  101.0
4 2024-01-01 00:00:00+00:00      B   50.0
5 2024-01-01 01:00:00+00:00      B   51.0
6 2024-01-01 02:00:00+00:00      B   52.0
8 2024-01-01 00:00:00+00:00      C   30.0


*When to use*: tidy data for plotting (seaborn loves long format), or ML pipelines that expect one row per observation.


### How do I aggregate while pivoting (e.g. mean of duplicates)?

`pivot_table(values=..., index=..., columns=..., aggfunc=...)`.


In [31]:
# Mean close per (ts, symbol) — handles duplicates.
pivot_table = df.pivot_table(values='close', index='ts', columns='symbol', aggfunc='mean')
print(pivot_table)

symbol                         A     B     C
ts                                          
2024-01-01 00:00:00+00:00  100.0  50.0  30.0
2024-01-01 01:00:00+00:00  101.0  51.0  30.5
2024-01-01 02:00:00+00:00    NaN  52.0  32.0


*Why not just `pivot`*: `pivot` raises on duplicates; `pivot_table` aggregates them. Use the explicit `aggfunc=` so the aggregation is documented.


---
## 9. Combining frames — merge, concat, align

How to combine multiple frames. The choice depends on whether you're joining on keys (`merge`), stacking (`concat`), or aligning by index (`align`).


### How do I join two frames on a column?

`df.merge(other, on='key', how='inner')`. Same as SQL JOIN.


In [32]:
prices = pd.DataFrame({'symbol': ['A', 'B', 'C'], 'price': [100, 50, 30]})
sectors = pd.DataFrame({'symbol': ['A', 'B', 'D'], 'sector': ['tech', 'fin', 'energy']})
print(prices.merge(sectors, on='symbol', how='left'))

  symbol  price sector
0      A    100   tech
1      B     50    fin
2      C     30    NaN


*Why explicit `how=`*: defaults change between pandas versions; specifying intent saves debugging. *Common mistake*: forgetting that 'inner' drops unmatched rows from BOTH sides — use 'left' if you want to preserve rows from the left frame.


### How do I stack two frames vertically (row-wise)?

`pd.concat([df1, df2], axis=0)`. Both frames need compatible columns.


In [33]:
top = df.head(3)
bot = df.tail(3)
print(pd.concat([top, bot]).reset_index(drop=True))

                         ts symbol  close  volume  close_centered
0 2024-01-01 00:00:00+00:00      A  100.0      10       -0.500000
1 2024-01-01 01:00:00+00:00      A  101.0      11        0.500000
2 2024-01-01 03:00:00+00:00      A    NaN      12             NaN
3 2024-01-01 00:00:00+00:00      C   30.0       8       -0.833333
4 2024-01-01 01:00:00+00:00      C   30.5       9       -0.333333
5 2024-01-01 02:00:00+00:00      C   32.0      10        1.166667


*Common mistake*: forgetting to `reset_index(drop=True)` and ending up with duplicate indices. This silently breaks `.loc` lookups later.


### How do I align two series with different indices?

`s1.align(s2, join='inner')` returns a tuple of aligned series, both reindexed to the join.


In [34]:
a = pd.Series([1, 2, 3], index=['x', 'y', 'z'])
b = pd.Series([10, 20], index=['x', 'y'])
a_aligned, b_aligned = a.align(b, join='inner')
print(a_aligned, '\n', b_aligned, sep='')

x    1
y    2
dtype: int64
x    10
y    20
dtype: int64


*When to use*: any time you do arithmetic between two series with potentially-different indices and want predictable behaviour. *Common mistake*: relying on pandas to broadcast — `a + b` produces NaN where indices don't match, easy to overlook.


---
## 10. DateTime — timestamps, timezones, indexing

Time-series work lives or dies on getting datetime types right. Always work in UTC; convert at the boundaries only.


### How do I parse a string column to datetime?

`pd.to_datetime(s, utc=True)`. Always specify UTC unless you have a strong reason not to.


In [35]:
s = pd.Series(['2024-01-01', '2024-01-02', '2024-01-03'])
print(pd.to_datetime(s, utc=True))

0   2024-01-01 00:00:00+00:00
1   2024-01-02 00:00:00+00:00
2   2024-01-03 00:00:00+00:00
dtype: datetime64[ns, UTC]


*Why UTC*: timezone-aware datetimes prevent subtle bugs when joining frames from different sources or running in different regions. *Common mistake*: parsing without `utc=True` and ending up with naive timestamps that silently break time-zone arithmetic.


### How do I get the hour / day / weekday of a datetime column?

Use the `.dt` accessor.


In [36]:
ts = pd.to_datetime(df['ts'])
print(ts.dt.hour.head())
print(ts.dt.day_name().head())

0    0
1    1
2    3
3    0
4    1
Name: ts, dtype: int32
0    Monday
1    Monday
2    Monday
3    Monday
4    Monday
Name: ts, dtype: object


*Common mistake*: `.dt.day` returns the day-of-month; `.dt.dayofyear` returns 1-365. Read the docs on which you actually want.


### How do I create a regular hourly grid?

`pd.date_range(start, end, freq='1h', tz='UTC')`.


In [37]:
grid = pd.date_range('2024-01-01', '2024-01-02', freq='1h', tz='UTC')
print(len(grid), 'hourly timestamps')

25 hourly timestamps


*When to use*: continuity checks (`series.reindex(grid)` reveals gaps as NaN), forward forecasting horizons.


### How do I check that an index is a continuous regular grid?

Build the expected `date_range` and compare.


In [38]:
df_idx = df.drop_duplicates(['ts', 'symbol']).set_index('ts').sort_index()
expected = pd.date_range(df_idx.index.min(), df_idx.index.max(), freq='1h', tz='UTC')
missing = expected.difference(df_idx.index.unique())
print(f'{len(missing)} missing hourly bars')

0 missing hourly bars


*Why*: silent gaps in the index break every rolling/diff operation downstream. Catch them at load time.


### How do I convert between timezones?

`series.dt.tz_convert('America/New_York')` for tz-aware; `.dt.tz_localize('UTC')` first if naive.


In [39]:
ts_utc = pd.to_datetime(['2024-01-01 12:00:00'], utc=True)
print(ts_utc.tz_convert('America/New_York'))

DatetimeIndex(['2024-01-01 07:00:00-05:00'], dtype='datetime64[ns, America/New_York]', freq=None)


*Common mistake*: calling `tz_convert` on a naive series — raises `TypeError`. Localize first, convert second.


---
## 11. Iteration and apply (and when to avoid both)

If you're iterating row-by-row in pandas, you're usually doing the wrong thing. The fast path is vectorised. Iteration is a last resort.


### How do I iterate over rows (and why you usually shouldn't)?

`itertuples()` is the fastest iteration. But ask yourself: is there a vectorised version first?


In [40]:
# Rarely-correct pattern.
for row in df.head(3).itertuples():
    print(row.symbol, row.close)

# Vectorised equivalent — what you usually want instead.
print(df.head(3).apply(lambda r: f"{r['symbol']}: {r['close']}", axis=1).tolist())

A 100.0
A 101.0
A nan
['A: 100.0', 'A: 101.0', 'A: nan']


*Why iteration is slow*: pandas overhead per row is enormous; vectorised ops happen in C. *When iteration is OK*: complex per-row logic that genuinely can't be vectorised (e.g. recursive state, calls to external APIs).


### How do I apply a function to every row?

`df.apply(func, axis=1)`. Slower than vectorising but cleaner than iteration.


In [41]:
df['summary'] = df.apply(lambda r: f"{r['symbol']}@{r['close']}", axis=1)
print(df[['symbol', 'close', 'summary']].head())

  symbol  close  summary
0      A  100.0  A@100.0
1      A  101.0  A@101.0
2      A    NaN    A@nan
3      B   50.0   B@50.0
4      B   51.0   B@51.0


*Common mistake*: using `apply` for arithmetic that could be vectorised — `df['a'] + df['b']` is 100× faster than `df.apply(lambda r: r['a'] + r['b'], axis=1)`.


### How do I create a new column based on a condition?

`np.where(cond, true_val, false_val)` for binary; `np.select` for multi-way.


In [42]:
import numpy as np
df['big'] = np.where(df['close'] > 50, 'big', 'small')
df['size'] = np.select(
    [df['close'] > 60, df['close'] > 30, df['close'] > 0],
    ['large', 'medium', 'small'],
    default='unknown',
)
print(df[['close', 'big', 'size']].head())

   close    big     size
0  100.0    big    large
1  101.0    big    large
2    NaN  small  unknown
3   50.0  small   medium
4   51.0    big   medium


*Why np.where over apply*: vectorised, much faster. *Why np.select for multi-way*: cleaner than nested `np.where` calls and easier to read.
